# 01 — Build & Load the Synthetic Dataset

Run once from the repo root before using any other notebook:

```
python -m src.retail_synth.run_all --config config/scenario_config.yaml
```

This generates every dimension and fact table under `data/raw/` (parquet) and loads
bronze → silver → gold into `data/warehouse/retail.duckdb`. `config/scenario_config.yaml`
is the single source of truth for scale and every scenario parameter — see
`dev_mode` there for a small fast dataset while iterating vs. the full enterprise-scale
build used for the actual demo.

This notebook doesn't re-run generation — it connects to the already-built warehouse
and reports the resulting scale against the config's declared targets, then runs the
same checks as `scripts/verify_build.py`.

In [1]:
import sys
sys.path.insert(0, "../src")
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from retail_synth.viz import CATEGORICAL, SEQUENTIAL_BLUE, DIVERGING, STATUS, style_fig

pd.options.display.float_format = "{:,.1f}".format
con = duckdb.connect("../data/warehouse/retail.duckdb", read_only=True)
AS_OF = con.execute("SELECT week_start_date FROM silver.dim_week WHERE is_as_of_week").fetchone()[0]
print(f"Connected. AS_OF date = {AS_OF}")

Connected. AS_OF date = 2025-12-01


## Scale vs. configured targets

In [2]:
import yaml
with open("../config/scenario_config.yaml") as f:
    raw_cfg = yaml.safe_load(f)
scale_key = "dev" if raw_cfg["dev_mode"] else "full"
target = raw_cfg["scale"][scale_key]
print(f"dev_mode = {raw_cfg['dev_mode']}  (scale profile: {scale_key})")
target

dev_mode = False  (scale profile: full)


{'n_styles': 1200, 'n_customers': 500000, 'n_suppliers': 60, 'n_campaigns': 45}

In [3]:
counts = {
    "stores": "SELECT COUNT(*) FROM silver.dim_store",
    "styles": "SELECT COUNT(*) FROM silver.dim_style",
    "skus": "SELECT COUNT(*) FROM silver.dim_sku",
    "customers": "SELECT COUNT(*) FROM silver.dim_customer",
    "weeks": "SELECT COUNT(*) FROM silver.dim_week",
    "sales_lines": "SELECT COUNT(*) FROM silver.fact_sales_line",
    "inventory_rows": "SELECT COUNT(*) FROM silver.fact_inventory_position",
    "returns_lines": "SELECT COUNT(*) FROM silver.fact_returns_line",
}
for label, sql in counts.items():
    print(f"{label:<16} {con.execute(sql).fetchone()[0]:>14,}")

stores                      280
styles                    1,200
skus                     21,619
customers               500,000
weeks                       156
sales_lines           5,514,181
inventory_rows       12,035,441
returns_lines         1,007,239


## Verification checks (same logic as `scripts/verify_build.py`)

In [4]:
import subprocess
result = subprocess.run(
    [sys.executable, "../scripts/verify_build.py"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

[ OK ] bronze: all 21 expected tables present
[ OK ] silver: all 21 expected tables present
[ OK ] gold: all 8 expected marts present
[ OK ] silver.fact_sales_line: 5,514,181 rows (within [3,000,000, 9,000,000])
[ OK ] silver.fact_inventory_position: 12,035,441 rows (within [8,000,000, 16,000,000])
[ OK ] silver.fact_returns_line: 1,007,239 rows (within [600,000, 2,000,000])
[ OK ] silver.fact_purchase_order_line: 4,743 rows (within [2,000, 10,000])
[ OK ] silver.fact_shipment_event: 18,957 rows (within [8,000, 35,000])
[ OK ] silver.fact_campaign_exposure: 385,925 rows (within [300,000, 800,000])
[ OK ] silver.fact_digital_engagement: 621,913 rows (within [300,000, 1,200,000])
[ OK ] fact_sales_line.sku_id -> dim_sku: no orphans
[ OK ] fact_sales_line.location_id -> dim_store: no orphans
[ OK ] fact_inventory_position.sku_id -> dim_sku: no orphans
[ OK ] fact_returns_line.sku_id -> dim_sku: no orphans
[ OK ] fact_inventory_position: no negative on_hand_units
[ OK ] weather shock: warm